## Search Engine with tools and agents

In [1]:
## Arxiv--Research
## Tools creation

from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [2]:
## Used the inbuilt tool of the wikipedia
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper = api_wrapper_wiki)
wiki.name

'wikipedia'

In [3]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper = api_wrapper_arxiv)
print(arxiv.name)

arxiv


In [4]:
tools = [wiki,arxiv]

In [5]:
## Custom tools[RAG Tools]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings


USER_AGENT environment variable not set, consider setting it to identify your requests.
c:\Users\User\anaconda3\envs\svenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
loader = WebBaseLoader("https://docs.langchain.com/langsmith/home")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb = FAISS.from_documents(documents,HuggingFaceEmbeddings())
retriever = vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000026D0B4833A0>, search_kwargs={})

In [7]:
from langchain_core.tools import create_retriever_tool
retriever_tool = create_retriever_tool(retriever,"langsmith-search","Search any information about langsmith")
retriever_tool.name

'langsmith-search'

In [8]:
tools=[wiki,arxiv,retriever_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\User\\anaconda3\\envs\\svenv\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 Tool(name='langsmith-search', description='Search any information about langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x0000026D0B474A60>, retriever=VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000026D0B48

In [16]:
## Run all the tools with agents and llm models

## Tools, LLM--> AgentExecutor
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
import os
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=os.getenv("GROQ_API_KEY"), model_name="llama-3.3-70b-versatile")



In [17]:
from langchain import hub
from langchain.agents import create_react_agent, AgentExecutor  # ✅ correct import

# ✅ Pull prompt correctly
prompt = hub.pull("hwchase17/react")
print(type(prompt))  # Should be <class 'langchain_core.prompts.prompt.PromptTemplate'>
print(prompt.input_variables)  # Should show ['agent_scratchpad', 'input', 'tool_names', 'tools']

# ✅ Create agent
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Test
agent_executor.invoke({"input": "What is LangSmith?"})

c:\Users\User\anaconda3\envs\svenv\lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
c:\Users\User\anaconda3\envs\svenv\lib\site-packages\langsmith\client.py:7195: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  prompt = loads(json.dumps(prompt_object.manifest))


<class 'langchain_core.prompts.prompt.PromptTemplate'>
['agent_scratchpad', 'input', 'tool_names', 'tools']


> Entering new AgentExecutor chain...
To answer this question, I need to find information about LangSmith. Since LangSmith seems to be related to language or AI, I should first try to search for it in a general knowledge database.

Action: langsmith-search
Action Input: What is LangSmithall available pages before exploring further.LangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, test prompts, and manage deployments all in one place, with your agent stack.

LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrati

{'input': 'What is LangSmith?',
 'output': 'LangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications, with features such as observability, evaluation, deployment, and platform setup. It provides a integrated workflow for building, debugging, and deploying AI applications, from local development to production, and meets the highest standards of data security and privacy with HIPAA, SOC 2 Type 2, and GDPR compliance.'}

In [18]:
agent_executor.invoke({"input":"What is ml"})



> Entering new AgentExecutor chain...
Thought: The question "What is ml" is quite broad, but it's likely referring to "Machine Learning" (ML), a field of study that gives computers the ability to learn without being explicitly programmed. To provide a more comprehensive answer, I should search for information on Machine Learning.

Action: wikipedia
Action Input: Machine LearningPage: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks witThought: The Wikipedia page provides a good overview of Machine Learning, including its definition, types, and applications. However, I'd like to know more about the current state of research in Machine Learning, including recent advancements and trends.

Action: arxiv
Action Input: Machine Learning recent advancements

c:\Users\User\anaconda3\envs\svenv\lib\site-packages\langchain_community\utilities\arxiv.py:105: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  ).results()


Published: 2023-06-07
Title: Changing Data Sources in the Age of Machine Learning for Official Statistics
Authors: Cedric De Boom, Michael Reusens
Summary: Data science has become increasingly essential for the production of official statistics, as iThought: The arxiv search results provide information on recent research papers related to Machine Learning, including its applications in official statistics. However, I'd like to know more about the tools and technologies used in Machine Learning, such as libraries and frameworks.

Action: langsmith-search
Action Input: Machine Learning tools and technologiesLangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresLLM GatewayPrivate betaOvervie

{'input': 'What is ml',
 'output': 'Machine Learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed. It has various applications, including official statistics, and is supported by tools and technologies such as LangSmith, a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Recent advancements in ML include its increasing importance in data science and its applications in various industries.'}